# 2.5D VoxelMorph V4 - fixed-grid FPGA training

This notebook trains the single shared 2.5D V4 core used by all orientations. Its only model input is **NHWC `[batch, 112, 96, 16]`**: eight moving slices followed by eight fixed slices.

The preprocessing contract is shared with calibration, local inference, and Vitis export:

- each volume is first resampled to canonical `[depth, height, width] = [96, 112, 96]` (trilinear for images, nearest neighbor for labels);
- axial: `112x96` without padding; coronal: `96x96` with centred top/bottom letterbox padding;
- sagittal: transpose `96x112` height/width to `112x96`;
- historical seven-slice windows: repeat the final slice to make the declared eight-slice contract.

Padding is masked out of the V4 Dice, MI, and smoothness losses.


In [ ]:
import os
from pathlib import Path
import json
import random
import sys

import numpy as np
import torch

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'train_2p5d_pytorchV4.py').exists():
    for candidate in (Path.cwd().resolve(), Path.cwd().resolve() / 'Voxelmorph' / '2.5D'):
        if (candidate / 'train_2p5d_pytorchV4.py').exists():
            NOTEBOOK_DIR = candidate
            break
    else:
        raise FileNotFoundError('Run from the repository root or Voxelmorph/2.5D directory.')

REPO_ROOT = NOTEBOOK_DIR.parents[1]
VOXELMORPH_ROOT = REPO_ROOT / 'Voxelmorph'
for candidate in (NOTEBOOK_DIR, REPO_ROOT, VOXELMORPH_ROOT):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import train_2p5d_pytorchV4 as v4_train
from vxm_2p5d_v4 import (
    INPUT_CHANNELS, INPUT_HEIGHT, INPUT_WIDTH, N_STACK, ORIENTATIONS,
    Vxm2p5dV4, combine_stacks, letterbox_stack,
)

print('Repository:', REPO_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## Configuration

Keep these values synchronized with `v4_model_spec.json`. Changing the input shape means creating a new model version and recompiling Vitis artifacts.


In [ ]:
DATA_ROOT = Path(os.environ.get('REGISTRATION_DATA_ROOT', str(REPO_ROOT / 'Data' / 'registration_dataset')))
MAX_TRAIN = None
MAX_VAL = None
CACHE_SUBJECTS = 32  # ~400 MB per split at the canonical grid; avoids raw-volume RAM pressure
CANONICAL_CACHE_DIR = DATA_ROOT / '.v4_canonical_cache'
BUILD_CANONICAL_CACHE = True  # one-time setup; later runs reuse these files

SEED = 42
BATCH_SIZE = 8
STEPS_PER_EPOCH = 200
VAL_STEPS = 50
EPOCHS = 1500
LEARNING_RATE = 1e-4
MI_WEIGHT = 0.5
SMOOTH_WEIGHT = 0.1
AUTO_RESUME = True

RUN_NAME = '2p5d_pt_v4_canonical'
WEIGHTS_DIR = VOXELMORPH_ROOT / 'trained_weights'
CHECKPOINT_DIR = WEIGHTS_DIR / '2p5d_pt_v4_canonical_checkpoints'
BEST_MODEL_PATH = WEIGHTS_DIR / '2p5d_dense_pt_v4_canonical_best.pth'
LATEST_STATE_PATH = CHECKPOINT_DIR / '2p5d_pt_v4_canonical_latest.pth'
CALIBRATION_DIR = NOTEBOOK_DIR / 'data' / 'calibration_data_v4_canonical'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for directory in (WEIGHTS_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('FPGA input:', (1, INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS))
print('Device:', DEVICE)
print('Best checkpoint:', BEST_MODEL_PATH)


## Validate the geometry contract

This is a fast guardrail. The central coronal content remains `96x96`; only its top and bottom borders are padded.


In [ ]:
orientation_shapes = {'axial': (112, 96), 'coronal': (96, 96), 'sagittal': (96, 112)}
for orientation, raw_hw in orientation_shapes.items():
    raw = np.zeros((N_STACK - 1, *raw_hw), dtype=np.float32)
    boxed, transform = letterbox_stack(raw, orientation)
    assert boxed.shape == (N_STACK, INPUT_HEIGHT, INPUT_WIDTH)
    print(f'{orientation:8s} raw={raw_hw} resized={transform.resized_hw} '
          f'padding=(top={transform.pad_top}, bottom={transform.pad_bottom}, '
          f'left={transform.pad_left}, right={transform.pad_right})')


## Load private dataset data

The dataset loader supplies images and segmentations. Image intensities are converted to `[-1, 1]` before the letterbox transform.


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if BUILD_CANONICAL_CACHE:
    v4_train.build_canonical_cache(DATA_ROOT, 'train', CANONICAL_CACHE_DIR, MAX_TRAIN)
    v4_train.build_canonical_cache(DATA_ROOT, 'val', CANONICAL_CACHE_DIR, MAX_VAL)

train_data = v4_train.load_split(DATA_ROOT, 'train', MAX_TRAIN, CACHE_SUBJECTS, CANONICAL_CACHE_DIR)
val_data = v4_train.load_split(DATA_ROOT, 'val', MAX_VAL, CACHE_SUBJECTS, CANONICAL_CACHE_DIR)
print('Training subjects:', len(train_data[0]), '| RAM-cached pairs:', CACHE_SUBJECTS)
print('Canonical disk cache:', CANONICAL_CACHE_DIR)
print('Validation subjects:', len(val_data[0]))
print('Example volume shape:', train_data[0][0].shape)


## Build a representative V4 batch

All orientations are sampled during training, but every output batch already has the fixed model shape.


In [ ]:
train_rng = np.random.default_rng(SEED)
val_rng = np.random.default_rng(SEED + 11)

example_batch = v4_train.build_batch(train_data[:2], train_data[2:], batch_size=2, rng=train_rng)
print('Orientation:', example_batch['orientation'])
print('Stored input (NCHW):', example_batch['input'].shape)
print('Model input (NHWC):', example_batch['input'].transpose(0, 2, 3, 1).shape)
print('Valid pixels per item:', example_batch['valid_mask'].sum(axis=(1, 2)))

val_batches = [
    v4_train.build_batch(val_data[:2], val_data[2:], batch_size=BATCH_SIZE, rng=val_rng)
    for _ in range(VAL_STEPS)
]


## Initialise or resume V4

V4 checkpoints are intentionally separate from V2/V3 checkpoints: its first layer is trained with the native 16-channel contract rather than compatibility padding.


In [ ]:
model = Vxm2p5dV4().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
start_epoch = 0
best_loss = float('inf')
history = {'train': [], 'val': []}

if AUTO_RESUME and LATEST_STATE_PATH.exists():
    state = torch.load(LATEST_STATE_PATH, map_location=DEVICE)
    if state.get('version') == 'v4-canonical-3d':
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        start_epoch = int(state['epoch'])
        best_loss = float(state['best_loss'])
        history = state.get('history', history)
        print(f'Resumed from epoch {start_epoch}; best validation loss={best_loss:.6f}')
    else:
        print('Ignoring non-V4 checkpoint:', LATEST_STATE_PATH)
else:
    print('Starting a new V4 run')

with torch.no_grad():
    flow = model(torch.from_numpy(example_batch['input']).permute(0, 2, 3, 1).to(DEVICE))
print('Flow output:', tuple(flow.shape))


## Train and validate

The loss matches the V4 script: segmentation Dice + masked mutual-information image loss + masked flow smoothness. The mask prevents letterbox borders from influencing optimization.


In [ ]:
model_spec = {
    'version': 'v4-canonical-3d',
    'input_shape': [1, INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS],
    'orientations': ORIENTATIONS,
    'pad_value': -1.0,
    'canonical_volume_shape': [96, 112, 96],
    'resize_policy': 'trilinear image / nearest-label resample to canonical 3D grid, then centred letterbox padding',
}
(CHECKPOINT_DIR / 'v4_model_spec.json').write_text(json.dumps(model_spec, indent=2), encoding='utf-8')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_rows = []
    for _ in range(STEPS_PER_EPOCH):
        batch = v4_train.build_batch(train_data[:2], train_data[2:], BATCH_SIZE, train_rng)
        train_rows.append(v4_train.run_step(model, optimizer, batch, DEVICE, MI_WEIGHT, SMOOTH_WEIGHT, train=True))
    train_metrics = v4_train.average(train_rows)

    model.eval()
    with torch.no_grad():
        val_rows = [
            v4_train.run_step(model, optimizer, batch, DEVICE, MI_WEIGHT, SMOOTH_WEIGHT, train=False)
            for batch in val_batches
        ]
    val_metrics = v4_train.average(val_rows)
    history['train'].append(train_metrics)
    history['val'].append(val_metrics)

    print(f'Epoch {epoch + 1}/{EPOCHS} | train={train_metrics["total"]:.6f} '
          f'val={val_metrics["total"]:.6f} dice={val_metrics["dice"]:.6f} '
          f'mi={val_metrics["mi"]:.6f} smooth={val_metrics["smooth"]:.6f}')

    if val_metrics['total'] < best_loss:
        best_loss = val_metrics['total']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print('Saved best model:', BEST_MODEL_PATH)

    torch.save({
        'version': 'v4-canonical-3d',
        'epoch': epoch + 1,
        'best_loss': best_loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'spec': model_spec,
    }, LATEST_STATE_PATH)

print('Best V4 checkpoint:', BEST_MODEL_PATH)
print('Best validation loss:', best_loss)


## Generate V4 calibration data

Run this after choosing the training data and before Vitis PTQ. It creates canonical-3D-resampled, letterboxed 16-channel tensors in the same layout used by V4 training.


In [ ]:
import subprocess

CALIBRATION_SAMPLES = 100
GENERATE_CALIBRATION = True  # set True after training / when ready

if GENERATE_CALIBRATION:
    subprocess.run([
        sys.executable,
        str(NOTEBOOK_DIR / 'prepare_calibration_data_v4.py'),
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(CALIBRATION_DIR),
        '--num-samples', str(CALIBRATION_SAMPLES),
        '--seed', str(SEED),
    ], check=True)
else:
    print('Calibration generation is disabled. Set GENERATE_CALIBRATION=True to run it.')


## Vitis quantization and compilation

This final step requires the Vitis AI Docker/runtime environment. It consumes the V4 checkpoint and the calibration data produced above. Do not enable it until the checkpoint and calibration directory exist.


In [ ]:
RUN_VITIS = True  # set True only in the host environment with Docker/Vitis AI configured

if RUN_VITIS:
    subprocess.run([sys.executable, str(REPO_ROOT / 'vitis_run_v4.py')], cwd=REPO_ROOT, check=True)
else:
    print('Vitis run is disabled. When ready, set RUN_VITIS=True or run:')
    print(f'  {sys.executable} {REPO_ROOT / "vitis_run_v4.py"}')
